# Évaluation du pipeline RAG avec RAGAS

**Objectif :** mesurer objectivement la qualité du pipeline via 4 métriques automatisées.

| Métrique | Ce qu'elle mesure | Données requises |
|---|---|---|
| `faithfulness` | La réponse reste-t-elle fidèle aux documents récupérés ? | question, answer, contexts |
| `answer_relevancy` | La réponse répond-elle bien à la question posée ? | question, answer, contexts |
| `context_precision` | Les chunks récupérés sont-ils tous utiles (pas de bruit) ? | question, contexts, ground_truth |
| `context_recall` | Les bons documents ont-ils été retrouvés ? | contexts, ground_truth |

Les 4 métriques utilisent Mistral comme **LLM-as-judge** — Mistral évalue lui-même la qualité des réponses.

Étapes :
1. Chargement de l'index et des clients
2. Définition du jeu de test (questions + réponses de référence)
3. Collecte des réponses RAG et contextes récupérés
4. Évaluation RAGAS
5. Analyse des résultats

In [1]:
import os
import pandas as pd
from datasets import Dataset
from langchain_community.vectorstores import FAISS
from langchain_mistralai import MistralAIEmbeddings
from langchain_mistralai.chat_models import ChatMistralAI
from mistralai.client import MistralClient
from mistralai.models.chat_completion import ChatMessage
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall

embed_model = MistralAIEmbeddings(
    model="mistral-embed",
    mistral_api_key=os.getenv("MISTRAL_API_KEY")
)
faiss_store = FAISS.load_local("../data/faiss_index", embed_model, allow_dangerous_deserialization=True)
client = MistralClient(api_key=os.getenv("MISTRAL_API_KEY"))

print(f"Index chargé : {faiss_store.index.ntotal} vecteurs")

Index chargé : 5028 vecteurs


## 1. Jeu de test annoté

RAGAS a besoin de deux choses annotées manuellement :
- `question` — une question représentative d'un vrai usage
- `ground_truth` — la réponse de référence qu'on attendrait idéalement

Les deux autres colonnes (`answer`, `contexts`) seront générées automatiquement par notre pipeline.

**Sur les `ground_truth`** : pour `context_precision` et `context_recall`, RAGAS vérifie si les contextes récupérés contiennent assez d'information pour produire la `ground_truth`. Ces réponses sont ici volontairement génériques — pour une évaluation rigoureuse en production, il faudrait les annoter à partir des vraies données.

In [2]:
test_questions = [
    "Quels concerts sont prévus à Rennes ?",
    "Y a-t-il des expositions gratuites à Rennes ?",
    "Quels événements sont proposés pour les enfants à Rennes ?",
    "Y a-t-il des spectacles de danse à Rennes ?",
    "Quels événements gratuits peut-on trouver à Rennes ?",
    "Y a-t-il des conférences ou ateliers culturels à Rennes ?",
    "Y a-t-il des événements musicaux en plein air à Rennes ?",
]

ground_truths = [
    "Plusieurs concerts sont prévus à Rennes avec des informations sur les dates, les lieux et les tarifs.",
    "Oui, il existe des expositions gratuites à Rennes dans des musées et centres d'art de la ville.",
    "Des événements familiaux et des activités pour enfants sont disponibles à Rennes.",
    "Des spectacles de danse sont organisés à Rennes dans différentes salles culturelles.",
    "De nombreux événements gratuits sont disponibles à Rennes dans différents lieux culturels.",
    "Des conférences et ateliers culturels sont organisés à Rennes.",
    "Des événements musicaux en plein air sont organisés à Rennes.",
]

print(f"{len(test_questions)} questions de test définies")

7 questions de test définies


## 2. Collecte des réponses et contextes

RAGAS a besoin de deux colonnes générées par le pipeline :
- `answer` — la réponse produite par Mistral
- `contexts` — la **liste des textes bruts** récupérés par FAISS (les `page_content`)

La fonction `ask_with_context()` est une version de `ask()` qui retourne les deux. Les `contexts` sont les corpus tels qu'indexés — c'est sur eux que RAGAS évalue faithfulness et context_recall.

In [ ]:
def ask(question: str, k: int = 5) -> tuple[str, list[str]]:
    """Pipeline RAG complet. Retourne (réponse générée, liste des corpus récupérés)."""
    docs = faiss_store.similarity_search(question, k=k)
    contexts = [doc.page_content for doc in docs]

    context_parts = []
    for doc in docs:
        m = doc.metadata
        context_parts.append(
            f"Titre : {m['title']}\n"
            f"Description : {doc.page_content}\n"
            f"Tarif : {m['conditions']}\n"
            f"Dates : {m['date_start']} → {m['date_end']}\n"
            f"Lieu : {m['location']}\n"
            f"Lien : {m['url']}"
        )
    context = "\n\n---\n\n".join(context_parts)

    prompt = f"""Tu es un assistant spécialisé dans les événements culturels à Rennes.
Réponds à la question en te basant uniquement sur les événements fournis ci-dessous.
Si l'information n'est pas dans le contexte, dis-le clairement.

ÉVÉNEMENTS :
{context}

QUESTION : {question}

RÉPONSE :"""

    response = client.chat(
        model="mistral-small-latest",
        messages=[ChatMessage(role="user", content=prompt)]
    )
    return response.choices[0].message.content, contexts


answers = []
contexts_list = []

for i, question in enumerate(test_questions):
    print(f"[{i+1}/{len(test_questions)}] {question}")
    answer, contexts = ask(question)
    answers.append(answer)
    contexts_list.append(contexts)

print("\nCollecte terminée.")

## 3. Construction du dataset RAGAS

RAGAS attend un objet `Dataset` de HuggingFace avec exactement 4 colonnes : `question`, `answer`, `contexts`, `ground_truth`. On assemble ici les données collectées.

In [4]:
data = pd.DataFrame({
    "question": test_questions,
    "answer": answers,
    "contexts": contexts_list,
    "ground_truth": ground_truths,
})

In [5]:
data

,question,answer,contexts,ground_truth
0,Quels concerts sont prévus à Rennes ?,Voici les concerts prévus à Rennes selon les é...,[Titre : Le concert du mardi - 𝗘́𝗹𝗶𝘀𝗮 𝗰𝗵𝗮𝗻𝘁𝗲 𝗝...,Plusieurs concerts sont prévus à Rennes avec d...
1,Y a-t-il des expositions gratuites à Rennes ?,"Oui, il y a des expositions gratuites à Rennes...",[Titre : Musées gratuits!\nDescription : Les m...,"Oui, il existe des expositions gratuites à Ren..."
2,Quels événements sont proposés pour les enfant...,Voici les événements proposés pour les enfants...,[Titre : Visite enfants : La chasse aux couleu...,Des événements familiaux et des activités pour...
3,Y a-t-il des spectacles de danse à Rennes ?,"Oui, il y a plusieurs spectacles de danse à Re...",[Titre : DANSES TOUS STYLES : LES RENCONTRES C...,Des spectacles de danse sont organisés à Renne...
4,Quels événements gratuits peut-on trouver à Re...,Voici les événements **gratuits** à Rennes men...,[Titre : Musées gratuits!\nDescription : Les m...,De nombreux événements gratuits sont disponibl...
5,Y a-t-il des conférences ou ateliers culturels...,"Oui, voici les conférences ou ateliers culture...",[Titre : Ateliers de conversation en français\...,Des conférences et ateliers culturels sont org...
6,Y a-t-il des événements musicaux en plein air ...,"Oui, il y a plusieurs événements musicaux en p...",[Titre : OPEN AIR - La Rennes des Voyous x Bre...,Des événements musicaux en plein air sont orga...


In [6]:
ragas_dataset = Dataset.from_pandas(data)

print(ragas_dataset)

Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 7
})


## 4. Évaluation RAGAS

RAGAS utilise un LLM pour évaluer automatiquement les réponses (**LLM-as-judge**). Par défaut il appelle OpenAI — on le reconfigure ici pour utiliser Mistral, cohérent avec notre stack.

> Cette cellule fait plusieurs appels API par question × par métrique.

In [7]:
ragas_llm = ChatMistralAI(
    model="mistral-small-latest",
    mistral_api_key=os.getenv("MISTRAL_API_KEY")
)

ragas_embeddings = MistralAIEmbeddings(
    model="mistral-embed",
    mistral_api_key=os.getenv("MISTRAL_API_KEY")
)

result = evaluate(
    ragas_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)

print(result)

Evaluating:   0%|          | 0/28 [00:00<?, ?it/s]

{'faithfulness': 0.9127, 'answer_relevancy': 0.8744, 'context_precision': 0.8143, 'context_recall': 1.0000}


## 5. Résultats et analyse

In [8]:
# Détail par question
df = result.to_pandas()
df["question"] = df["question"].str[:55]
display(df[["question", "faithfulness", "answer_relevancy", "context_precision", "context_recall"]].round(3))

,question,faithfulness,answer_relevancy,context_precision,context_recall
0,Quels concerts sont prévus à Rennes ?,1.000,0.851,0.867,1.0
1,Y a-t-il des expositions gratuites à Rennes ?,1.000,0.844,1.000,1.0
2,Quels événements sont proposés pour les enfant...,0.889,0.893,0.000,1.0
3,Y a-t-il des spectacles de danse à Rennes ?,1.000,0.866,1.000,1.0
4,Quels événements gratuits peut-on trouver à Re...,1.000,0.850,1.000,1.0
5,Y a-t-il des conférences ou ateliers culturels...,1.000,0.894,0.917,1.0
6,Y a-t-il des événements musicaux en plein air ...,0.500,0.922,0.917,1.0


### Investigation des scores anormaux

Deux questions ont des métriques inhabituellement basses :
- **Q2 "enfants"** → `context_precision = 0.000` : RAGAS juge que les 5 documents récupérés ne sont pas utiles pour répondre
- **Q6 "plein air"** → `faithfulness = 0.500` : Mistral a affirmé quelque chose d'absent des contextes récupérés

In [11]:
df_full = result.to_pandas()

for idx, label in [(2, "Q2 — enfants (context_precision=0.000)"), (6, "Q6 — plein air (faithfulness=0.500)")]:
    row = df_full.iloc[idx]
    print(f"\n{'='*60}")
    print(f"{label}")
    print(f"{'='*60}")
    print(f"\n[RÉPONSE GÉNÉRÉE]\n{row['answer']}")
    print(f"\n[CONTEXTES RÉCUPÉRÉS]")
    for i, ctx in enumerate(row["contexts"]):
        print(f"\n  — Doc {i+1} : {ctx}")


Q2 — enfants (context_precision=0.000)

[RÉPONSE GÉNÉRÉE]
Voici les événements proposés pour les enfants à Rennes, basés sur les informations fournies :

1. **Visite enfants : La chasse aux couleurs**
   - **Date** : 13 septembre 2025, de 9h à 10h
   - **Lieu** : Musée Quai Zola, 20 Quai Emile Zola - 35000 Rennes
   - **Inscription obligatoire** : Envoyer un email à *mba-reservations@ville-rennes.fr* avec les détails demandés.
   - **Gratuit** (15 enfants max. + 1 accompagnateur par enfant).

2. **Visite enfants : Raconte-moi une histoire effrayante**
   - **Date** : 4 octobre 2025, de 9h à 10h
   - **Lieu** : Musée Quai Zola, 20 Quai Emile Zola - 35000 Rennes
   - **Inscription obligatoire** : Même processus que ci-dessus.
   - **Gratuit**.

3. **Stage Fabrique "À l'École des beaux-arts de Bretagne, les enfants font le carnaval"**
   - **Dates** : 13, 14 et 15 avril 2026 (de 13h30 à 17h45 le 13/04, de 9h à 13h les 14/04 et 15/04)
   - **Lieu** : EESAB-site de Rennes, 34 rue Hoche, 35

### Diagnostic

**Q2 "enfants" — `context_precision = 0.000` : faux négatif de RAGAS**

Le retrieval est **parfait** — les 5 documents sont tous des événements pour enfants à Rennes (visites musée, jeux de piste, stage artistique). Le problème vient de notre `ground_truth` trop générique : *"Des événements familiaux et des activités pour enfants sont disponibles à Rennes."*

`context_precision` mesure si chaque contexte est utile pour produire la `ground_truth`. Quand la ground_truth est une phrase vague sans détails concrets, le LLM-as-judge ne peut pas établir le lien → score à zéro par défaut.

**Correction** : réécrire la ground_truth avec des détails ancrés dans les données, ex. *"Des visites guidées pour enfants au musée (gratuit, sur inscription), des jeux de piste et des stages artistiques sont proposés à Rennes."*

---

**Q6 "plein air" — `faithfulness = 0.500` : faux négatif de RAGAS**

Après inspection des contextes complets, tous les détails fournis par Mistral sont bien présents dans les documents récupérés :
- "Parc du Thabor, Esplanade Charles de Gaulle, CCN" → présents dans le Doc 2 (description complète de Rennes fête la Musique)
- "14h à 23h" → présents dans les métadonnées du corpus (Date de début/fin)
- "Disco, House, House progressive" → présents dans le Doc 1

C'est le LLM-as-judge (Mistral-small) qui a mal évalué, probablement déstabilisé par la longueur et la densité du Doc 2. Les Docs 3 et 5 (*"Histoires en plein air"*) sont des événements de conte récupérés à cause du mot "plein air", mais Mistral a correctement choisi de les ignorer.

**Leçon** : RAGAS lui-même peut produire des faux négatifs avec un LLM-as-judge de taille modeste sur des contextes très longs. Pour une évaluation plus fiable, on pourrait passer à `mistral-large` comme juge, ou vérifier manuellement les cas limites.